# Chapter 38: Multimodal AI Systems
**Artificial Intelligence Learning Hub — Level 4**
*Pengembang: Aji Pamungkas Tri Nurcahyo — Institut Teknologi Sumatera*

Notebook pendamping ini mempraktikkan integrasi dan penyelarasan representasi lintas modalitas (*Image + Text*) secara 100% offline pada CPU:
1. **Paired Synthetic Image–Text Dataset:** 320 pasang latih & 96 pasang uji tak terlihat (4 warna $\times$ 4 bentuk).
2. **Encoders:** `TinyImageEncoder` (CNN 4x4 spasial) & `TinyTextEncoder` (token embedding).
3. **Multimodal Fusion:** Komparasi *Early*, *Late*, dan *Cross-Attention* beserta ketahanan terhadap *missing modalities*.
4. **Contrastive Learning & Miniature Dual Encoder (CLIP-style):** Penyelarasan ruang bersama $d=32$ dengan *Symmetric InfoNCE*.
5. **Bidirectional Cross-Modal Retrieval:** Evaluasi `Recall@1/3/5` dua arah (*Image-to-Text* & *Text-to-Image*).


In [2]:
import sys, os
import torch, numpy as np

sys.path.append(os.path.abspath("../code"))

from paired_dataset_generator import generate_paired_splits, PairedMultimodalDataset, COLOR_NAMES, SHAPES, encode_caption_ids
from cross_attention_fusion import numpy_cross_attention_worked_example, train_and_evaluate_cross_attention
from early_fusion import train_and_evaluate_early_fusion
from late_fusion import train_and_evaluate_late_fusion
from contrastive_loss import manual_infonce_worked_example
from clip_miniature import train_clip_miniature
from cross_modal_evaluation import evaluate_i2t, evaluate_t2i

print(f"PyTorch: {torch.__version__} | NumPy: {np.__version__} | Modul Chapter 38 siap.")


PyTorch: 2.15.0+cpu | NumPy: 2.5.4 | Modul Chapter 38 siap.


## 1. Eksplorasi Dataset Pasangan Citra–Teks
Memverifikasi pemisahan ketat *train* (`320` pasangan) vs *test* (`96` pasangan dengan variasi ukuran, posisi, serta derau baru).


In [4]:
train_items, test_items = generate_paired_splits(seed=42)
train_ds = PairedMultimodalDataset(train_items)
test_ds = PairedMultimodalDataset(test_items)

print(f"Train Split: {len(train_ds)} pairs | Images: {tuple(train_ds.images.shape)}")
print(f"Test  Split: {len(test_ds)}  pairs | Tokens: {tuple(test_ds.tokens.shape)}")
print(f"Sample [0] : '{train_ds.captions[0]}' -> Tokens {train_ds.tokens[0].tolist()}")


Train Split: 320 pairs | Images: (320, 3, 32, 32)
Test  Split: 96  pairs | Tokens: (96, 2)
Sample [0] : 'red circle' -> Tokens [1, 5]


## 2. Perbandingan Strategi Fusi & Ketahanan Modalitas Hilang
Menguji *Early Fusion* (concat), *Late Fusion* (weighted probability ensemble), dan *Cross-Attention Fusion* (Q=Text, K/V=Image Patches).


In [6]:
_, ef_b, ef_nt, ef_ni = train_and_evaluate_early_fusion(epochs=20, seed=42)
_, lf_b, lf_nt, lf_ni = train_and_evaluate_late_fusion(epochs=20, seed=42)
_, ca_b, ca_nt, ca_ni = train_and_evaluate_cross_attention(epochs=20, seed=42)

print(f"Early Fusion    -> Both: {ef_b:.2f}% | Img Only: {ef_nt:.2f}% | Txt Only: {ef_ni:.2f}%")
print(f"Late Fusion     -> Both: {lf_b:.2f}% | Img Only: {lf_nt:.2f}% | Txt Only: {lf_ni:.2f}%")
print(f"Cross-Attention -> Both: {ca_b:.2f}% | Img Only: {ca_nt:.2f}% | Txt Only: {ca_ni:.2f}%")


Early Fusion    -> Both: 100.00% | Img Only: 25.00% | Txt Only: 93.75%
Late Fusion     -> Both: 100.00% | Img Only: 43.75% | Txt Only: 100.00%
Cross-Attention -> Both: 100.00% | Img Only: 25.00% | Txt Only: 62.50%


## 3. Verifikasi Cross-Attention & InfoNCE Contrastive Loss
Memeriksa paritas perhitungan tensor kecil NumPy vs PyTorch untuk *Scaled Dot-Product Cross-Attention* dan *Symmetric InfoNCE Loss*.


In [8]:
numpy_cross_attention_worked_example()
manual_infonce_worked_example()


1. Worked Tensor Cross-Attention Verification (NumPy == PyTorch Parity):
   Q (Text Tokens)  shape: (2, 4)   | K, V (Image Patches) shape: (3, 4)
   Attention Weights (2x3): [[0.6792, 0.0919, 0.2289], [0.0919, 0.6792, 0.2289]]
   Row Sums == 1.0 Check  : [1.0, 1.0] (PASS)
   Fused Output     shape: (2, 4)   (PASS: Not mere concatenation!)

1. Worked Manual 3x3 Contrastive Matrix Verification (tau=0.5):
   - Manual Image->Text Loss : 0.72219
   - Manual Text->Image Loss : 0.73931
   - Symmetric Total Loss    : 0.73075 (Matches PyTorch 0.73075!)


## 4. Pelatihan Miniature Dual Encoder & Evaluasi Retrieval Silang
Melatih `MiniatureDualEncoder` (36,704 param) selama 35 epoch menggunakan *unique-concept batches*, lalu mengukur `Recall@1/3/5` pada seluruh 96 citra uji tak terlihat.


In [10]:
clip_model, _, test_ds, losses = train_clip_miniature(epochs=35, seed=42, save_ckpt=False)
canon_caps = [f"{c} {s}" for c in COLOR_NAMES for s in SHAPES]
canon_toks = torch.tensor([encode_caption_ids(cap) for cap in canon_caps], dtype=torch.long)

with torch.no_grad():
    img_embs = clip_model.encode_image(test_ds.images)
    txt_embs = clip_model.encode_text(canon_toks)

i2t = evaluate_i2t(img_embs, test_ds.concept_ids, txt_embs)
t2i = evaluate_t2i(txt_embs, img_embs, test_ds.concept_ids)

print(f"InfoNCE Loss: Epoch 1 = {losses[0]:.4f} -> Epoch 35 = {losses[-1]:.4f}")
print(f"Image -> Text Retrieval (N=96): R@1={i2t['R@1']}%, R@3={i2t['R@3']}%, MedianRank={i2t['median_rank']}")
print(f"Text -> Image Retrieval (N=16): R@1={t2i['R@1']}%, P@6={t2i['mean_P@6']}%, MedianRank={t2i['median_rank']}")


InfoNCE Loss: Epoch 1 = 1.6808 -> Epoch 35 = 0.0717
Image -> Text Retrieval (N=96): R@1=100.0%, R@3=100.0%, MedianRank=1.0
Text -> Image Retrieval (N=16): R@1=100.0%, P@6=100.0%, MedianRank=1.0


## 5. Kesimpulan & Batasan Ilmiah
1. **Ketahanan Fusi:** *Late Fusion* mempertahankan performa fallback yang jauh lebih baik ketika modalitas teks hilang (`43.75%`) atau citra hilang (`100.00%`) dibandingkan penggabungan rapat tanpa *modality dropout*.
2. **Penyelarasan Kontrastif:** *Unique-concept batch sampling* menghindari benturan *false negatives* pada InfoNCE dan mendorong margin kesamaan pasangan positif-negatif hingga `+1.0403` (`+0.9753` vs `-0.0650`).
3. **Batasan:** Skor 100% dicapai pada korpus geometris sintetis terkontrol (`16` konsep warna–bentuk); prototipe miniatur ini tidak memiliki pemahaman dunia nyata setara bobot CLIP resmi yang dilatih pada 400 juta pasangan.
